In [ ]:
import os, sys
# walk up from the current directory to the repo root (the one holding `aima/`)
_root = os.path.abspath(os.getcwd())
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'aima')):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

# THE VACUUM WORLD   

In this notebook, we will be discussing **the structure of agents** through an example of the **vacuum agent**. The job of AI is to design an **agent program** that implements the agent function: the mapping from percepts to actions. We assume this program will run on some sort of computing device with physical sensors and actuators: we call this the **architecture**:

<h3 align="center">agent = architecture + program</h3>

## Setup

In [ ]:
# Imports
from aima.agents import *
from aima.notebook_utils import psource

In [ ]:
# Helper class/functions for the environment
class DirectionType(type):
    """Fancy python trickery that allows us to use either
        >> Direction.Right
        or
        >> Direction["Right"]
    and get the same outcome (1,0)
    """
    Right = (1,0)
    Left  = (-1,0)
    Down  = (0,-1)
    Up    = (0,1)
    
    def __getitem__(self, key):
        match key.lower():
            case "right": return self.Right
            case "left": return self.Left
            case "down": return self.Down
            case "up": return self.Up
            case _: return (0,0) # default case

class Directions(metaclass=DirectionType):
    """A simple helper class for moving around a 2d grid."""
    pass

all_literal_directions = ["Right","Left","Up","Down"]

def add_direction(position, direction):
    if direction in all_literal_directions:
        direction_offset = Directions[direction]
    else:
        direction_offset = direction
    return (position[0] + direction_offset[0], position[1] + direction_offset[1])

The actual environment

In [ ]:
grid4x4 = [
    (x,y)
    for x in range(4)
    for y in range(4)
]
gridLiteral = [
    [0, 3, 2, 0,],
    [0, 0, 0, 0,],
    [0, 4, 6, 8,],
    [7, 9, 0, 10,],
]
gridValues = [
    [gridLiteral[3-y][x] for x in range(4)]
    for y in range(4)
] # Flips the gridLiteral so you can index it like [x][y] instead of having to do [3-y][x]
# Thus gridValues[0][0] is the bottom left, and gridValues[3][3] is the top right (the starting point)
startingLocation = (3,3)

In [ ]:

class VacuumEnvironment2d(TrivialVacuumEnvironment):

    history: list[tuple[int,int]] = []

    def __init__(self):
        super().__init__()
        # We're overriding the status they set here
        self.history.append(self.default_location(None))
        self.status = {
            location: gridValues[location[0]][location[1]]
            for location in grid4x4
        }

    def execute_action(self, agent, action):
        """Change agent's location and/or location's status; track performance.
        Score 10 for each dirt cleaned; -1 for each move."""
        
        agent.performance -= 1
        destination = add_direction(agent.location, Directions[action])
        # Only move if it is to a location inside the grid
        if destination in self.status:
            agent.location = destination
            self.history.append(destination)

    def default_location(self, thing):
        """Agents start in either location at random."""
        return startingLocation

    def displayPathing(self):
        emptyGrid = [
                [ "  -  " for _ in range(4) ]
                for _ in range(4)
                ]
        numMoves = len(self.history)
        # Gets a list numbering 0,1,2, etc..
        steps = [ f"  {str(i)}  " for i in range(numMoves) ]
        steps[0] =  "start"
        steps[-1] = " end "
        print(self.history)
        for i in range(numMoves):
            x,y = self.history[i][0], self.history[i][1]
            emptyGrid[x][y] = steps[i]

        for row in emptyGrid[::-1]:
            print(row)

## Q1 - Random Agent Program

In [ ]:
e = VacuumEnvironment2d()

e.history=[(3,3),(3,2),(3,1),(2,1)]

print(e.status)
e.displayPathing()